## Exemple de représentation de tarifs

In [1]:
import sys
from pathlib import Path

# remonte à la racine du projet OCPI
sys.path.append(str(Path().resolve().parents[0]))

In [2]:
import json
from source.tariff import Tariff, TariffElement, TariffElements, TariffDimensionType, TariffRestrictions, PriceComponent
from source.utils import Format, Price

## Exemple du modèle OCPI 2.3 avec différents formats
Trois formats sont interchangeables et peuvent être utilisés pour construire un objet Tariff:
- format JSON OCPI
- format JSON simplifié
- format texte réduit (moins de 256 caractères)

## Exemple 1 : tarif simple 
Tarif de 0.30 €/kWh

In [3]:
tariff_text = "EN30"

tariff = Tariff.from_string(tariff_text, id="t1", tariff_alt_text="exemple de tarif simple")

In [4]:
print(f"tarif json simplifié : \n{json.dumps(tariff.to_json(ocpi=False, simple=True), indent=2)}")

tarif json simplifié : 
{
  "id": "t1",
  "elements": [
    {
      "price_components": {
        "ENERGY": 0.3
      }
    }
  ],
  "tariff_alt_text": "exemple de tarif simple",
  "last_updated": "2026-04-27T23:53:30.460557"
}


In [5]:
print(f"tarif json Qualicharge : \n{json.dumps(tariff.to_json(ocpi=False), indent=2)}")

tarif json Qualicharge : 
{
  "id": "t1",
  "elements": [
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.3
        }
      ]
    }
  ],
  "tariff_alt_text": "exemple de tarif simple",
  "last_updated": "2026-04-27T23:53:30.460557"
}


In [6]:
print(f"tarif json OCPI 2.3 : \n{json.dumps(tariff.to_json(ocpi=True), indent=2)}")

tarif json OCPI 2.3 : 
{
  "id": "t1",
  "elements": [
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.3
        }
      ]
    }
  ],
  "type": "AD_HOC_PAYMENT",
  "tariff_alt_text": "exemple de tarif simple",
  "last_updated": "2026-04-27T23:53:30.460557",
  "tax_included": "YES"
}


## Exemple 2 : tarif jour/nuit

In [7]:
tariff_text = "EN30+T>07:00+T<20:00|EN20"

tariff = Tariff.from_string(tariff_text, id="t2", tariff_alt_text="exemple de tarif jour-nuit")

In [8]:
print(f"tarif json simplifié : \n{json.dumps(tariff.to_json(ocpi=False, simple=True), indent=2)}")

tarif json simplifié : 
{
  "id": "t2",
  "elements": [
    {
      "price_components": {
        "ENERGY": 0.3
      },
      "restrictions": {
        "start_time": "07:00",
        "end_time": "20:00"
      }
    },
    {
      "price_components": {
        "ENERGY": 0.2
      }
    }
  ],
  "tariff_alt_text": "exemple de tarif jour-nuit",
  "last_updated": "2026-04-27T23:53:30.522798"
}


In [9]:
print(f"tarif json Qualicharge : \n{json.dumps(tariff.to_json(ocpi=False), indent=2)}")

tarif json Qualicharge : 
{
  "id": "t2",
  "elements": [
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.3
        }
      ],
      "restrictions": {
        "start_time": "07:00",
        "end_time": "20:00"
      }
    },
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.2
        }
      ]
    }
  ],
  "tariff_alt_text": "exemple de tarif jour-nuit",
  "last_updated": "2026-04-27T23:53:30.522798"
}


In [10]:
print(f"tarif json OCPI 2.3 : \n{json.dumps(tariff.to_json(ocpi=True), indent=2)}")

tarif json OCPI 2.3 : 
{
  "id": "t2",
  "elements": [
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.3
        }
      ],
      "restrictions": {
        "start_time": "07:00",
        "end_time": "20:00"
      }
    },
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.2
        }
      ]
    }
  ],
  "type": "AD_HOC_PAYMENT",
  "tariff_alt_text": "exemple de tarif jour-nuit",
  "last_updated": "2026-04-27T23:53:30.522798",
  "tax_included": "YES"
}


## Exemple 3 : tarif jour/nuit avec parking
"*entre 08:00 et 20:00 : 0.4525€ par kwh de charge, 6.0€ par heure d'occupation hors charge, entre 20:00 et 08:00 : 0.4525€ par kwh de charge*"


In [11]:
tariff_text = "EN45+PT600+T>08:00+T<20:00|EN35"

tariff = Tariff.from_string(tariff_text, id="t2", tariff_alt_text="exemple de tarif jour-nuit avec parking")

In [12]:
print(f"tarif json Qualicharge : \n{json.dumps(tariff.to_json(ocpi=False, simple=False), indent=2)}")

tarif json Qualicharge : 
{
  "id": "t2",
  "elements": [
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.45
        },
        {
          "type": "PARKING_TIME",
          "price": 6.0
        }
      ],
      "restrictions": {
        "start_time": "08:00",
        "end_time": "20:00"
      }
    },
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.35
        }
      ]
    }
  ],
  "tariff_alt_text": "exemple de tarif jour-nuit avec parking",
  "last_updated": "2026-04-27T23:53:30.569129"
}


In [13]:
#exemple de création d'un tarif avec les entités OCPI
price_component1 = PriceComponent(type=TariffDimensionType.ENERGY, price=Price(amount=0.30))
price_component2 = PriceComponent(type=TariffDimensionType.FLAT, price=Price(amount=1.20))
price_component3 = PriceComponent(type=TariffDimensionType.ENERGY, price=Price(amount=0.10))
price_component4 = PriceComponent(type=TariffDimensionType.FLAT, price=Price(amount=1.50))
tariff_element1 = TariffElement(
        price_components=[price_component1, price_component2],
        restrictions=TariffRestrictions.from_json({"days_of_week": ["MONDAY", "TUESDAY"],"start_date": "2024-01-01"})
    )
tariff_element2 = TariffElement(price_components=[price_component3, price_component4])
tariff_elements = TariffElements([tariff_element1, tariff_element2])
tariff = Tariff(id="tariff1", elements=tariff_elements)


In [14]:
# Affichage du tarif au format JSON OCPI 2.3
print(json.dumps(tariff.to_json(), indent=2))

{
  "id": "tariff1",
  "elements": [
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.3
        },
        {
          "type": "FLAT",
          "price": 1.2
        }
      ],
      "restrictions": {
        "days_of_week": [
          "MONDAY",
          "TUESDAY"
        ],
        "start_date": "2024-01-01"
      }
    },
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.1
        },
        {
          "type": "FLAT",
          "price": 1.5
        }
      ]
    }
  ],
  "type": "AD_HOC_PAYMENT",
  "last_updated": "2026-04-27T23:53:30.594772",
  "tax_included": "YES"
}


In [15]:
# Affichage du tarif au format JSON simplifié
print(json.dumps(tariff.to_json(simple=True), indent=2))

{
  "id": "tariff1",
  "elements": [
    {
      "price_components": {
        "ENERGY": 0.3,
        "FLAT": 1.2
      },
      "restrictions": {
        "days_of_week": [
          "MONDAY",
          "TUESDAY"
        ],
        "start_date": "2024-01-01"
      }
    },
    {
      "price_components": {
        "ENERGY": 0.1,
        "FLAT": 1.5
      }
    }
  ],
  "type": "AD_HOC_PAYMENT",
  "last_updated": "2026-04-27T23:53:30.594772",
  "tax_included": "YES"
}


In [16]:
# Affichage du tarif dans un format texte compacté (uniquement la partie elements)
tariff_text = tariff.to_string()
tariff_text

'EN30+FL120+J=LuMa+D>2024-01-01|EN10+FL150'

In [17]:
# Affichage du tarif au format OCPI 2.3 à partir de la partie texte compacté uniquement
print(json.dumps(Tariff.convert(tariff_text, Format.JSON_OCPI), indent=2))

{
  "id": "undefined",
  "elements": [
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.3
        },
        {
          "type": "FLAT",
          "price": 1.2
        }
      ],
      "restrictions": {
        "days_of_week": [
          "MONDAY",
          "TUESDAY"
        ],
        "start_date": "2024-01-01"
      }
    },
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.1
        },
        {
          "type": "FLAT",
          "price": 1.5
        }
      ]
    }
  ],
  "type": "AD_HOC_PAYMENT",
  "last_updated": "2026-04-27T23:53:30.660818",
  "tax_included": "YES"
}


## Exemple d'un tarif variable suivant les tranches horaires

- 0,22 €/kWh de 00:00 à 04:00
- 0,22 €/kWh de 04:00 à 08:00
- 0,49 €/kWh de 08:00 à 20:00
- 0,35 €/kWh de 20:00 à 00:00
- Frais de congestion (parking sans recharge) 0,50 €/min

In [18]:
tariff_text = """
  EN22 + PT3000 + T>00:00 + T<04:00
  EN22 + PT3000 + T>04:00 + T<08:00
  EN49 + PT3000 + T>08:00 + T<20:00
  EN35 + PT3000"""

tariff = Tariff.from_string(tariff_text, id="t1", tariff_alt_text="exemple de tarif par plage horaire")


In [19]:
print(f"tarif texte compact (sur une ligne) : \n{tariff.to_string()} - {len(tariff.to_string())} caractères")

tarif texte compact (sur une ligne) : 
EN22+PT3000+T>00:00+T<04:00|EN22+PT3000+T>04:00+T<08:00|EN49+PT3000+T>08:00+T<20:00|EN35+PT3000 - 95 caractères


In [20]:
print(f"tarif json (compact) : \n{json.dumps(tariff.to_json(simple=True), indent=2)}")

tarif json (compact) : 
{
  "id": "t1",
  "elements": [
    {
      "price_components": {
        "ENERGY": 0.22,
        "PARKING_TIME": 30.0
      },
      "restrictions": {
        "start_time": "00:00",
        "end_time": "04:00"
      }
    },
    {
      "price_components": {
        "ENERGY": 0.22,
        "PARKING_TIME": 30.0
      },
      "restrictions": {
        "start_time": "04:00",
        "end_time": "08:00"
      }
    },
    {
      "price_components": {
        "ENERGY": 0.49,
        "PARKING_TIME": 30.0
      },
      "restrictions": {
        "start_time": "08:00",
        "end_time": "20:00"
      }
    },
    {
      "price_components": {
        "ENERGY": 0.35,
        "PARKING_TIME": 30.0
      }
    }
  ],
  "type": "AD_HOC_PAYMENT",
  "tariff_alt_text": "exemple de tarif par plage horaire",
  "last_updated": "2026-04-27T23:53:30.682717",
  "tax_included": "YES"
}


In [21]:
print(f"tarif json (OCPI 2.3) : \n{json.dumps(tariff.to_json(), indent=2)}")

tarif json (OCPI 2.3) : 
{
  "id": "t1",
  "elements": [
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.22
        },
        {
          "type": "PARKING_TIME",
          "price": 30.0
        }
      ],
      "restrictions": {
        "start_time": "00:00",
        "end_time": "04:00"
      }
    },
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.22
        },
        {
          "type": "PARKING_TIME",
          "price": 30.0
        }
      ],
      "restrictions": {
        "start_time": "04:00",
        "end_time": "08:00"
      }
    },
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.49
        },
        {
          "type": "PARKING_TIME",
          "price": 30.0
        }
      ],
      "restrictions": {
        "start_time": "08:00",
        "end_time": "20:00"
      }
    },
    {
      "price_components": [
        {
          "type": "ENER

In [22]:
print(tariff.to_text())

Tariff : "t1"
- ENERGY :
  - 0.22 €/kWh à partir de 00:00 et jusqu'à 04:00
  - 0.22 €/kWh à partir de 04:00 et jusqu'à 08:00
  - 0.49 €/kWh à partir de 08:00 et jusqu'à 20:00
  - 0.35 €/kWh sinon
- PARKING_TIME :
  - 30.0 €/h



In [45]:
import pandas as pd

with open('query_result_2026-05-04T22_12_28.80864901+02_00.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
extras = [json.loads(day['extras']) for day in data]
pd_dat = pd.DataFrame(data)
for i in range(len(pd_dat)):
    ext = pd.DataFrame(extras[i])
    print(ext['energy'].sum(), pd_dat['energy'][i])
print(pd_dat['energy'].sum())

407.356934 407.356934
309.251197 309.251197
301.906114 301.906114
315.615115 315.615115
312.20977400000004 312.20977400000004
315.67140900000004 315.67140900000004
354.35601599999995 354.35601599999995
347.278856 347.278856
315.431906 315.431906
342.145797 342.145797
342.274096 342.274096
334.347827 334.347827
350.17136500000004 350.17136500000004
378.402478 378.402478
410.71889 410.71889
359.120021 359.120021
365.242629 365.242629
338.32273699999996 338.32273699999996
349.5209 349.5209
385.877637 385.877637
374.268232 374.268232
346.597938 346.597938
371.08394699999997 371.08394699999997
354.481496 354.481496
391.96913 391.96913
418.605243 418.605243
458.479829 458.479829
424.36570600000005 424.36570600000005
373.675827 373.675827
385.220717 385.220717
388.253194 388.253194
11222.222956999998


In [53]:
ext0 = pd.DataFrame(extras[25])
for i in range(len(ext0)):
     if len(ext0['id_station_itinerance'][i]) < 8:
          print(ext0['id_station_itinerance'][i])

In [56]:
stations = set()
for i in range(len(pd_dat)):
    ext = pd.DataFrame(extras[i])
    stations |= set(ext['id_station_itinerance'])
print(len(stations))

740


In [58]:
print(sorted(stations))

['FRELCP10056589', 'FRELCP10153067', 'FRELCP10192581', 'FRELCP10271525', 'FRELCP10271539', 'FRELCP10271540', 'FRELCP10271545', 'FRELCP10271566', 'FRELCP10281487', 'FRELCP10408432', 'FRELCP10476408', 'FRELCP10484667', 'FRELCP10503935', 'FRELCP10545373', 'FRELCP10647727', 'FRELCP10653312', 'FRELCP10785290', 'FRELCP10785293', 'FRELCP11027595', 'FRELCP11054365', 'FRELCP11272286', 'FRELCP11285191', 'FRELCP11335279', 'FRELCP11357916', 'FRELCP11395926', 'FRELCP11442750', 'FRELCP11559913', 'FRELCP11559914', 'FRELCP11569602', 'FRELCP11569603', 'FRELCP11569618', 'FRELCP11576789', 'FRELCP11610155', 'FRELCP11901153', 'FRELCP11916145', 'FRELCP11925448', 'FRELCP11925453', 'FRELCP11933057', 'FRELCP11943433', 'FRELCP11943564', 'FRELCP11943569', 'FRELCP11943570', 'FRELCP11943571', 'FRELCP12004504', 'FRELCP12004508', 'FRELCP12010864', 'FRELCP12010865', 'FRELCP12093888', 'FRELCP12224359', 'FRELCP12320515', 'FRELCP12496135', 'FRELCP12496136', 'FRELCP12531793', 'FRELCP12577838', 'FRELCP12826415', 'FRELCP12

In [64]:
ext0 = pd.DataFrame(extras[30])
ext0.loc[ext0['id_station_itinerance']=='FRELCP12954725']

,code,siren,energy,entity,id_station_itinerance
359,FRELC,891624884,0.659386,Electra,FRELCP12954725
